# Session 18: Ethical Considerations in AI

**Applied Machine Learning Using Python**  
**Duration**: 4 Hours (TL18)  
**Level**: 🔴 Advanced

---

## 📋 Objectives
1. Define Bias in AI (Historical and Proxy Bias).
2. Execute Mathematical Audits (Disparate Impact ratio).
3. Apply mitigation techniques safely.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries Loaded Successfully.")

---
## §1 — Analyzing Proxy Bias in Loan Generation
You work for a massive global bank. A new "Smart AI" was just built to approve or deny loans instantly based on zip code and income.

Before releasing it, the Ethics Team demands you audit it for Racial Bias. The original Data Scientists say: *"It's impossible for it to be racist. We deleted the 'Race' column from the training data before we gave it to the algorithm."*

You don't trust them. Let's look at the data.

In [ ]:
# Synthesize Mock banking data
np.random.seed(42)
n = 1000
race_code = np.random.binomial(1, 0.5, n) # 1 = Majority, 0 = Minority

# Here is the problem: In this simulated city, Zipcode is heavily tied to race due to historical redlining.
# Zipcode 90210 is 90% Majority. Zipcode 90001 is 90% Minority.
zipCode = np.where(race_code == 1, np.random.choice([90210, 90001], p=[0.9, 0.1], size=n),
                                   np.random.choice([90210, 90001], p=[0.1, 0.9], size=n))

# Income is generated. Let's make it identical for both races to prove the AI isn't finding income differences.
income = np.random.normal(50000, 10000, n)

# The biased outcome: Historical bankers rejected 90001 unfairly.
prob_approval = np.where(zipCode == 90210, 0.8, 0.2)
loan_approved = np.random.binomial(1, prob_approval)

df = pd.DataFrame({'ZipCode': zipCode, 'Income': income, 'Race': race_code, 'Approved': loan_approved})
display(df.head())

### Training the "Race Blind" Algorithm
Watch what happens when we delete Race from the data, but keep ZipCode.

In [ ]:
X_blind = df[['ZipCode', 'Income']]
y = df['Approved']

model = RandomForestClassifier(max_depth=3, random_state=42)
model.fit(X_blind, y)

df['AI_Decision'] = model.predict(X_blind)
print("AI Model Trained without knowing Race!")

---
## §2 — The Ethical Audit (Disparate Impact)
We must now compare the AI's decisions against the Race column it wasn't allowed to see.

**The Four-Fifths Rule:** The minority approval rate must be at least 80% that of the majority.

In [ ]:
maj_approval_rate = df[df['Race'] == 1]['AI_Decision'].mean()
min_approval_rate = df[df['Race'] == 0]['AI_Decision'].mean()

print(f"Majority AI Approval Rate: {maj_approval_rate * 100:.1f}%")
print(f"Minority AI Approval Rate: {min_approval_rate * 100:.1f}%")

disparate_impact = min_approval_rate / maj_approval_rate
print(f"\nDisparate Impact Ratio: {disparate_impact:.3f}")

if disparate_impact < 0.80:
    print("\n❌ AUDIT FAILED.")
    print("Explanation: Because ZipCode was perfectly correlated with Race, taking away the 'Race' column did not magically make the AI fair. It just used ZipCode as a 'Proxy' to discriminate anyway.")

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(x=['Minority', 'Majority'], y=[min_approval_rate, maj_approval_rate], palette='Reds')
plt.title("Proof of Proxy Bias in Loan Algorithm")
plt.ylabel("AI Approval Rate")
plt.show()

## Conclusion
To fix this, you must retrain the AI using ONLY `Income`. If you use ZipCode, the model remains racist. This is what ML Engineers do every day!